In [ ]:
# 제거할 컬럼 목록 -> drop
drop_cols = [
    "host_name",
    "neighborhood_overview",  # neighbourhood와 의미 상충
    "host_identity_verified",  # host_verifications와 중복
    "has_availability",  # 단일값
    "host_about",  # 결측 40% + 중요도 낮음
    "host_neighbourhood",  # host_location 하위그룹, 중복
    "minimum_nights_avg_ntm",  # 제거
    "maximum_nights_avg_ntm",  # 제거
]

# 실제 존재하는 컬럼만 제거
df_cleaned = df_cleaned.drop(columns=drop_cols)

print(f"제거한 컬럼 수: {len(drop_cols)}개")
print(f"제거 후 shape: {df_cleaned.shape}")

In [ ]:
# host_response_time 결측 -> 최빈값
mode_val = df_cleaned["host_response_time"].mode()[0]
df_cleaned["host_response_time"] = df_cleaned["host_response_time"].fillna(mode_val)

# host_response_rate / host_acceptance_rate
# 퍼센트 문자열 -> float 변환, 중앙값
for col in ["host_response_rate", "host_acceptance_rate"]:
    if df_cleaned[col].dtype == object:
        df_cleaned[col] = (
            df_cleaned[col].astype(str).str.replace("%", "", regex=False).replace("nan", np.nan).astype(float) / 100
        )
    missing = df_cleaned[col].isna().sum()
    median_val = df_cleaned[col].median()
    df_cleaned[col] = df_cleaned[col].fillna(median_val)
    print(f"{col} 결측 {missing}개 -> 중앙값({median_val:.2f})")

# host_location -> 결측 5283개 4로
# (이미 인코딩된 경우 가정, 원본 문자열이면 별도 처리 필요)
if "host_location" in df_cleaned.columns:
    missing = df_cleaned["host_location"].isna().sum()
    df_cleaned["host_location"] = df_cleaned["host_location"].fillna(4)
    print(f"host_location 결측 {missing}개를 -> 4")

# host_is_superhost / host_has_profile_pic
bool_cols = ["host_is_superhost", "host_has_profile_pic"]

for col in bool_cols:
    mode_val = df_cleaned[col].mode()[0]
    missing = df_cleaned[col].isna().sum()
    df_cleaned[col] = df_cleaned[col].fillna(mode_val)
    print(f"{col} 결측 {missing}개 -> 최빈값({mode_val})")

In [ ]:
# host_listings_count 결측 -> 최빈값
mode_val = df_cleaned["host_listings_count"].mode()[0]
df_cleaned["host_listings_count"] = df_cleaned["host_listings_count"].fillna(mode_val)


# host_total_listings_count 결측 -> 최빈값
mode_val = df_cleaned["host_total_listings_count"].mode()[0]
df_cleaned["host_total_listings_count"] = df_cleaned["host_total_listings_count"].fillna(mode_val)

In [ ]:
# bedrooms / beds / bathrooms 결측 -> 중앙값
for col in ["bedrooms", "beds", "bathrooms"]:
    if col in df_cleaned.columns:
        missing = df_cleaned[col].isna().sum()
        median_val = df_cleaned[col].median()
        df_cleaned[col] = df_cleaned[col].fillna(median_val)
        print(f"{col} 결측 {missing}개 → 중앙값({median_val:.1f})")

In [ ]:
# first_review, last_review 파생 변수 생성 후 결측 처리, 원본은 드랍
base_date = pd.Timestamp("2025-03-02")

# 파생변수 생성: 처음 리뷰를 받은 날부터 센 기간, 마지막 리뷰를 받은 날부터 센 기간
# days_since_last_review가 작으면? = 최근에도 리뷰를 받고 있다 = 잘 운영중인 숙소다
df_cleaned["days_since_first_review"] = (base_date - df_cleaned["first_review"]).dt.days
df_cleaned["days_since_last_review"] = (base_date - df_cleaned["last_review"]).dt.days

for col in ["days_since_first_review", "days_since_last_review"]:
    missing = df_cleaned[col].isna().sum()
    max_val = df_cleaned[col].max()
    df_cleaned[col] = df_cleaned[col].fillna(max_val)
    print(f"{col} 결측 {missing}개 -> 최댓값({max_val:.0f}일)")

# 원본 리뷰 날짜 컬럼은 drop
df_cleaned = df_cleaned.drop(columns=["first_review", "last_review"])

In [ ]:
remaining = df_cleaned.isna().sum()
remaining = remaining[remaining > 0].sort_values(ascending=False)

if len(remaining) == 0:
    print("\n결측치 없음 — 모델링 준비 끝!")
else:
    print(f"\n결측치 있는 컬럼 ({len(remaining)}개):")
    print(remaining)

print(f"\n모델링 전 df_cleaned shape: {df_cleaned.shape}")

In [ ]:
# 얘네는 이진처리
# license                      17845
# description                  405
# bathrooms_text
# neighbourhood                10046